In [7]:
project_root = Path("/content/drive/MyDrive/CCTV_VLM_Project")

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

os.chdir(project_root)

NameError: name 'os' is not defined

In [8]:
import cv2
import numpy as np
import pandas as pd
import supervision as sv
import torch
import time
from pathlib import Path

from src.config import (
    MODELS_DIR,
    VIDEOS_DIR,
    OUTPUTS_DIR,
    CLASS_MAPPING,
    DINO_PROMPT,
    YOLO_CONF_THRESH,
    DINO_BOX_THRESH,
    DINO_TEXT_THRESH,
    NMS_IOU_THRESH
)

from src.yolo_detector import YoloDetector
from src.dino_detector import DinoDetector
from src.fusion import BoxFusion

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Supervision:", sv.__version__)

PyTorch: 2.11.0+cu130
CUDA available: True
Supervision: 0.31.4


In [9]:
video_path = VIDEOS_DIR / "warehouse2.mp4"
yolo_weight_path = MODELS_DIR / "yolo11s_warehouse_best.pt"

assert video_path.exists(), f"Video not found: {video_path}"
assert yolo_weight_path.exists(), f"YOLO weights not found: {yolo_weight_path}"

yolo = YoloDetector(
    model_path=str(yolo_weight_path),
    conf=YOLO_CONF_THRESH
)

dino = DinoDetector(
    box_thresh=DINO_BOX_THRESH,
    text_thresh=DINO_TEXT_THRESH
)

fusion = BoxFusion(
    iou_threshold=NMS_IOU_THRESH
)

print("YOLO, Grounding DINO and fusion initialized.")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/978 [00:00<?, ?it/s]

YOLO, Grounding DINO and fusion initialized.


In [14]:
video_info = sv.VideoInfo.from_video_path(str(video_path))

duration = (
    video_info.total_frames / video_info.fps
    if video_info.fps > 0 else 0
)

print("Video:", video_path.name)
print("Resolution:", video_info.resolution_wh)
print("FPS:", video_info.fps)
print("Total frames:", video_info.total_frames)
print(f"Duration: {duration:.2f} seconds")

Video: warehouse2.mp4
Resolution: (1920, 1080)
FPS: 24.8
Total frames: 173
Duration: 6.98 seconds


In [15]:
detection_records = []

yolo_times = []
dino_times = []
fusion_times = []

print("Detection evaluation initialized.")

Detection evaluation initialized.


In [16]:
for frame_idx, frame in enumerate(
    sv.get_video_frames_generator(source_path=str(video_path))
):
    timestamp = frame_idx / video_info.fps

    # YOLO
    start = time.perf_counter()
    yolo_det = yolo.detect(frame)
    yolo_times.append(time.perf_counter() - start)

    # Grounding DINO
    start = time.perf_counter()
    dino_det = dino.detect(frame, prompt=DINO_PROMPT)
    dino_times.append(time.perf_counter() - start)

    # Fusion
    start = time.perf_counter()
    fused_det = fusion.merge_and_suppress(yolo_det, dino_det)
    fusion_times.append(time.perf_counter() - start)

    # Store each detection separately
    for source, detections in [
        ("yolo", yolo_det),
        ("grounding_dino", dino_det),
        ("fused", fused_det)
    ]:
        if len(detections) == 0:
            continue

        for box, confidence, class_id in zip(
            detections.xyxy,
            detections.confidence,
            detections.class_id
        ):
            detection_records.append({
                "frame_id": frame_idx,
                "timestamp": timestamp,
                "detector_source": source,
                "class_id": int(class_id),
                "class_name": CLASS_MAPPING[int(class_id)],
                "confidence": float(confidence),
                "x1": float(box[0]),
                "y1": float(box[1]),
                "x2": float(box[2]),
                "y2": float(box[3])
            })

    if (frame_idx + 1) % 10 == 0:
        print(
            f"Processed {frame_idx + 1}/{video_info.total_frames} frames"
        )

print("Inference completed!")

Processed 10/173 frames
Processed 20/173 frames
Processed 30/173 frames
Processed 40/173 frames
Processed 50/173 frames
Processed 60/173 frames
Processed 70/173 frames
Processed 80/173 frames
Processed 90/173 frames
Processed 100/173 frames
Processed 110/173 frames
Processed 120/173 frames
Processed 130/173 frames
Processed 140/173 frames
Processed 150/173 frames
Processed 160/173 frames
Processed 170/173 frames
Inference completed!


In [17]:
df = pd.DataFrame(detection_records)

print("Total detection records:", len(df))

display(
    df.groupby(["detector_source", "class_name"])
      .size()
      .unstack(fill_value=0)
)

display(df.head(10))

Total detection records: 2185


class_name,forklift,pallet_truck,person
detector_source,,,
fused,15,332,606
grounding_dino,15,332,562
yolo,0,0,323


,frame_id,timestamp,detector_source,class_id,class_name,confidence,x1,y1,x2,y2
0,0,0.0,yolo,0,person,0.717720,689.137573,861.102600,797.125916,1077.317017
1,0,0.0,yolo,0,person,0.391509,469.459137,433.131683,536.042725,530.948120
2,0,0.0,grounding_dino,0,person,0.625279,1466.927612,407.098602,1520.426270,481.704285
3,0,0.0,grounding_dino,0,person,0.556513,476.104980,458.372253,531.024048,531.559692
4,0,0.0,grounding_dino,0,person,0.432098,694.839966,860.556274,797.984009,1075.647461
5,0,0.0,grounding_dino,2,pallet_truck,0.471610,1309.997681,468.576141,1598.032104,800.944336
6,0,0.0,grounding_dino,2,pallet_truck,0.426405,135.109985,467.392487,709.060669,1076.364136
7,0,0.0,grounding_dino,0,person,0.385985,1599.220947,448.474396,1641.302246,534.624695
8,0,0.0,grounding_dino,0,person,0.352947,342.976288,411.777740,391.169861,499.163452
9,0,0.0,grounding_dino,0,person,0.363913,698.607544,860.966187,797.086914,1006.226562


In [18]:
csv_path = OUTPUTS_DIR / "warehouse2_detection_results.csv"

df.to_csv(csv_path, index=False)

print("Saved:", csv_path)
print("Rows:", len(df))

Saved: /content/drive/MyDrive/CCTV_VLM_Project/outputs/warehouse2_detection_results.csv
Rows: 2185


In [19]:
frame_counts = (
    df.groupby(["frame_id", "detector_source"])
      .size()
      .unstack(fill_value=0)
      .reset_index()
)

display(frame_counts.head(15))

print("\nDetection count statistics:")
display(frame_counts.drop(columns="frame_id").describe())

detector_source,frame_id,fused,grounding_dino,yolo
0,0,7,8,2
1,1,5,5,2
2,2,5,5,2
3,3,6,6,2
4,4,6,6,2
5,5,6,7,2
6,6,7,7,2
7,7,5,5,1
8,8,5,5,2
9,9,6,6,1



Detection count statistics:


detector_source,fused,grounding_dino,yolo
count,173.000000,173.000000,173.000000
mean,5.508671,5.254335,1.867052
std,0.937702,0.990681,0.799499
min,4.000000,3.000000,1.000000
25%,5.000000,5.000000,1.000000
50%,5.000000,5.000000,2.000000
75%,6.000000,6.000000,2.000000
max,9.000000,8.000000,5.000000


In [20]:
print(f"YOLO average: {np.mean(yolo_times)*1000:.2f} ms/frame")
print(f"DINO average: {np.mean(dino_times)*1000:.2f} ms/frame")
print(f"Fusion average: {np.mean(fusion_times)*1000:.2f} ms/frame")

total_avg = np.mean(yolo_times) + np.mean(dino_times) + np.mean(fusion_times)

print(f"Combined average: {total_avg*1000:.2f} ms/frame")
print(f"Estimated processing FPS: {1/total_avg:.2f}")

YOLO average: 25.62 ms/frame
DINO average: 609.32 ms/frame
Fusion average: 0.61 ms/frame
Combined average: 635.55 ms/frame
Estimated processing FPS: 1.57
